# Diccionario de datos

**Proyecto:** Relación entre condiciones meteorológicas y calidad del aire en ciudades de Latinoamérica

Este documento describe las variables de las dos fuentes utilizadas y del dataset integrado que resulta de unirlas.

**Unidad de análisis:** cada fila del dataset integrado representa la medición diaria de una ciudad, es decir, la combinación ciudad × día.

**Periodo cubierto por el dataset integrado:** 2023-01-01 a 2024-04-20.

## 1. Fuente A — LA_daily_climate.csv (Kaggle)

**Origen:** Latin America Weather and Air Quality Data, Kaggle  
**Formato:** CSV  
**Granularidad:** diaria por ciudad  
**Dimensiones originales:** 31,440 filas × 14 columnas  
**Cobertura:** 20 ciudades · 20 países  
**Periodo original:** 2020-01-01 a 2024-04-20
**URL:** https://www.kaggle.com/datasets/anycaroliny/latin-america-weather-and-air-quality-data/data


### Variables climáticas

| Variable | Tipo de dato | Descripción | Unidad |
|---|---|---|---|
| `country` | object | País al que pertenece la ciudad | — |
| `city` | object | Nombre de la ciudad | — |
| `date` | datetime | Fecha de la medición | UTC |
| `latitude` | float64 | Latitud de la ciudad | grados |
| `longitude` | float64 | Longitud de la ciudad | grados |
| `temperature_2m_max` | float64 | Temperatura máxima diaria a 2 metros | °C |
| `temperature_2m_min` | float64 | Temperatura mínima diaria a 2 metros | °C |
| `temperature_2m_mean` | float64 | Temperatura media diaria a 2 metros | °C |
| `apparent_temperature_max` | float64 | Temperatura aparente máxima diaria | °C |
| `apparent_temperature_min` | float64 | Temperatura aparente mínima diaria | °C |
| `apparent_temperature_mean` | float64 | Temperatura aparente media diaria | °C |
| `precipitation_sum` | float64 | Precipitación acumulada durante el día | mm |
| `wind_speed_10m_max` | float64 | Velocidad máxima del viento a 10 metros | km/h |
| `et0_fao_evapotranspiration` | float64 | Evapotranspiración de referencia | mm |

### Observaciones de la Fuente A

La fuente climática contiene información diaria para 20 ciudades de Latinoamérica. El periodo original disponible va del 2020-01-01 al 2024-04-20.

Para la integración con los datos de calidad del aire se utiliza únicamente el periodo comprendido entre 2023-01-01 y 2024-04-20, con el objetivo de trabajar con un periodo común entre ambas fuentes.

Las variables meteorológicas principales que se utilizarán posteriormente para analizar su relación con la calidad del aire son temperatura, precipitación y velocidad del viento.

## 2. Fuente B: Datos de calidad del aire

**Fuente:** Open-Meteo Air Quality API  
**Origen:** https://open-meteo.com/en/docs/air-quality-api  
**Formato original:** JSON  
**Método de obtención:** API mediante solicitudes HTTP utilizando Python y la librería `requests`  
**Frecuencia original:** Horaria  
**Frecuencia utilizada:** Diaria  
**Periodo utilizado:** 2023-01-01 a 2024-04-20  
**Número de ciudades:** 20

Los datos de calidad del aire fueron obtenidos mediante la API de Open-Meteo utilizando las coordenadas de las ciudades presentes en la fuente climática. La extracción se realizó mediante el script `src/ingestion/obtener_calidad_aire.py`.

No fue necesario utilizar una API key para esta extracción.

### Variables de calidad del aire

| Variable | Tipo de dato | Descripción | Unidad | Valores faltantes |
|---|---|---|---|---|
| `time` | datetime | Fecha y hora de la medición | UTC | 0% |
| `pm10` | float64 | Concentración de partículas PM10 | μg/m³ | 0% |
| `pm2_5` | float64 | Concentración de partículas PM2.5 | μg/m³ | 0% |
| `carbon_monoxide` | float64 | Concentración de monóxido de carbono | μg/m³ | 0% |
| `nitrogen_dioxide` | float64 | Concentración de dióxido de nitrógeno | μg/m³ | 0% |
| `sulphur_dioxide` | float64 | Concentración de dióxido de azufre | μg/m³ | 0% |
| `ozone` | float64 | Concentración de ozono | μg/m³ | 0% |

### Variables agregadas durante la extracción

Además de las variables obtenidas directamente de la API, se agregaron las siguientes variables para identificar cada registro:

| Variable | Tipo de dato | Descripción |
|---|---|---|
| `city` | object | Ciudad correspondiente a las coordenadas consultadas |
| `country` | object | País correspondiente a la ciudad |
| `latitude` | float64 | Latitud utilizada en la consulta a la API |
| `longitude` | float64 | Longitud utilizada en la consulta a la API |
| `date` | datetime | Fecha obtenida a partir de la variable `time` |

La variable `date` se utilizó posteriormente para convertir los datos horarios de calidad del aire en promedios diarios.

## 3. Transformación de los datos de calidad del aire

Los datos obtenidos mediante la API tienen una frecuencia horaria. Debido a que la fuente climática trabaja con información diaria, se calcularon promedios diarios para cada contaminante.

Para cada ciudad y cada fecha se calcularon los promedios de:

- PM10
- PM2.5
- Monóxido de carbono
- Dióxido de nitrógeno
- Dióxido de azufre
- Ozono

De esta manera, los datos de calidad del aire pueden integrarse con la información climática diaria.

Una limitación de esta transformación es que los promedios diarios pueden ocultar picos de contaminación que ocurran durante determinadas horas del día.

## 4. Periodo utilizado

Durante las pruebas realizadas con la API se encontraron valores faltantes para algunas consultas correspondientes a 2022. Por este motivo, se decidió utilizar para la extracción final el periodo comprendido entre:

**2023-01-01 y 2024-04-20**

Este periodo permite trabajar con datos disponibles y utilizar un periodo común con la fuente climática.

## 5. Integración de las fuentes

Los datos climáticos y de calidad del aire se integraron utilizando las siguientes variables:

**`date + city + country + latitude + longitude`**

La fecha por sí sola no era suficiente para realizar la unión, ya que existen registros correspondientes a diferentes ciudades en una misma fecha. Utilizar únicamente `date` podía generar combinaciones incorrectas entre ciudades y datos de calidad del aire.

Por esta razón, se utilizó la fecha junto con la información de ubicación para asegurar que los datos meteorológicos y de calidad del aire correspondieran a la misma ciudad.

## 6. Dataset integrado

El dataset integrado tiene las siguientes dimensiones:

**9,520 filas × 20 columnas**

Estas dimensiones corresponden a:

**476 fechas × 20 ciudades = 9,520 registros**

Cada fila representa una combinación de una ciudad y un día, incluyendo variables meteorológicas y de calidad del aire.

El dataset integrado constituye la base para los análisis posteriores sobre la relación entre las condiciones meteorológicas y los niveles de contaminación del aire.

## 7. Diagnóstico inicial

Como parte de la preparación de los datos se revisaron:

- Valores faltantes.
- Registros duplicados.
- Tipos de datos.
- Periodos disponibles.
- Número de ciudades.
- Correspondencia entre ciudades y coordenadas.

Después de la extracción final de los datos de calidad del aire se obtuvieron **9,520 registros**, correspondientes a 20 ciudades y 476 fechas, sin valores faltantes ni duplicados en la combinación `date + city`.

Los datos fueron convertidos a formatos compatibles y las fechas se trabajaron utilizando UTC.

## 8. Limitaciones de los datos

Los datos de calidad del aire obtenidos mediante Open-Meteo corresponden a datos modelados y no directamente a mediciones realizadas por estaciones de monitoreo.

Además, la conversión de datos horarios a promedios diarios puede ocultar variaciones o picos de contaminación que ocurran durante determinadas horas.

Otra limitación es que las dos fuentes tienen diferentes periodos originales, por lo que fue necesario seleccionar un periodo común para realizar la integración.

## 9. Variables derivadas previstas

Para las siguientes etapas del proyecto se considera crear variables derivadas que permitan realizar un análisis más completo, entre ellas:

| Variable | Descripción |
|---|---|
| `month` | Mes correspondiente a la fecha |
| `season` | Temporada del año |
| `is_rainy_day` | Indicador de si hubo precipitación durante el día |
| `temp_range` | Diferencia entre la temperatura máxima y mínima |

Estas variables todavía no forman parte del dataset integrado final de este entregable y serán utilizadas en las siguientes etapas del proyecto.

## 10. Conclusión

El conjunto de datos integrado combina información meteorológica y de calidad del aire para 20 ciudades de Latinoamérica durante el periodo 2023-01-01 a 2024-04-20.

La estructura obtenida permite analizar posteriormente posibles relaciones entre variables como temperatura, precipitación y viento con contaminantes como PM2.5, PM10, CO, NO2, SO2 y O3.

Este diccionario documenta las fuentes, variables, transformaciones y características principales de los datos utilizados en el proyecto.